# 01 — Tanager load and preprocess

Loads a Tanager Basic Radiance scene, converts it to surface reflectance, previews it as true-color and false-color composites, and selects a vegetated region of interest (ROI) with a soil/shadow mask.

**Input:** a Tanager Basic Radiance scene downloaded per `data/README.md`. No scene is bundled with this repository.

Uses PROSPECT-D/4SAIL-relevant spectral ranges downstream (see `REFERENCES.md`); this notebook only handles I/O and preprocessing.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent))
from src import tanager_io, atmospheric, channel_selector

## Scene path

Point this at your locally downloaded scene (see `data/README.md`). Not committed to the repository.

In [ ]:
# TODO: set to your downloaded scene, e.g. Path("../data/tanager_scene_01/scene.hdr")
SCENE_PATH = Path("../data/tanager_scene_01/scene.hdr")
SCENE_FORMAT = "envi"  # "envi" or "geotiff"

## Load radiance cube

In [ ]:
if not SCENE_PATH.exists():
    raise FileNotFoundError(
        f"No scene found at {SCENE_PATH}. Download a Tanager Basic Radiance "
        "scene per data/README.md and update SCENE_PATH above."
    )

if SCENE_FORMAT == "envi":
    radiance, wavelengths = tanager_io.load_envi_radiance(SCENE_PATH)
elif SCENE_FORMAT == "geotiff":
    # Wavelengths must come from the scene's accompanying metadata file.
    raise NotImplementedError("Supply band-centre wavelengths for GeoTIFF input.")
else:
    raise ValueError(f"Unknown SCENE_FORMAT: {SCENE_FORMAT}")

print(f"Radiance cube shape: {radiance.shape}")
print(f"Bands: {len(wavelengths)}, range {wavelengths.min():.0f}-{wavelengths.max():.0f} nm")

## Convert to surface reflectance

Uses the empirical line method if reference targets are available in the scene, otherwise the simplified atmospheric correction in `src/atmospheric.py`. This is a general open approach, not SilvaIQ's internal atmospheric correction pipeline.

In [ ]:
# TODO: if the scene includes known reference targets (calibration panels,
# invariant surfaces), populate reference_radiance / reference_reflectance
# and use atmospheric.empirical_line_correction instead.
SOLAR_ZENITH_DEG = 30.0  # TODO: from scene metadata

reflectance = atmospheric.simplified_atmospheric_correction(
    radiance, wavelengths, solar_zenith_deg=SOLAR_ZENITH_DEG
)
print(f"Reflectance cube shape: {reflectance.shape}")

## RGB and false-color preview

In [ ]:
rgb_bands = channel_selector.nearest_band_indices(
    wavelengths, {"red": 650, "green": 560, "blue": 450}
)
false_color_bands = channel_selector.nearest_band_indices(
    wavelengths, {"nir": 840, "red": 650, "green": 560}
)


def stretch(cube_slice, p_low=2, p_high=98):
    lo, hi = np.percentile(cube_slice, [p_low, p_high])
    return np.clip((cube_slice - lo) / (hi - lo), 0, 1)


rgb = np.stack(
    [reflectance[:, :, rgb_bands[c]] for c in ("red", "green", "blue")], axis=-1
)
false_color = np.stack(
    [reflectance[:, :, false_color_bands[c]] for c in ("nir", "red", "green")], axis=-1
)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(stretch(rgb))
axes[0].set_title("True color (R-G-B)")
axes[0].axis("off")
axes[1].imshow(stretch(false_color))
axes[1].set_title("False color (NIR-Red-Green)")
axes[1].axis("off")
plt.tight_layout()
plt.savefig("../results/figures/01_scene_preview.png", dpi=150)
plt.show()

## Vegetated ROI selection and soil/shadow mask

Uses NDVI thresholding as a first-pass vegetation mask, and a brightness threshold to exclude shadow. Refine interactively per scene.

In [ ]:
nir = reflectance[:, :, false_color_bands["nir"]]
red = reflectance[:, :, rgb_bands["red"]]
ndvi = (nir - red) / (nir + red + 1e-6)

NDVI_VEG_THRESHOLD = 0.3   # TODO: tune per scene
SHADOW_BRIGHTNESS_THRESHOLD = 0.05  # TODO: tune per scene

brightness = reflectance.mean(axis=-1)
vegetation_mask = (ndvi > NDVI_VEG_THRESHOLD) & (brightness > SHADOW_BRIGHTNESS_THRESHOLD)

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
im = axes[0].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
axes[0].set_title("NDVI")
axes[0].axis("off")
plt.colorbar(im, ax=axes[0], fraction=0.046)
axes[1].imshow(vegetation_mask, cmap="gray")
axes[1].set_title("Vegetation mask (soil/shadow excluded)")
axes[1].axis("off")
plt.tight_layout()
plt.savefig("../results/figures/01_roi_mask.png", dpi=150)
plt.show()

print(f"Vegetated pixels: {vegetation_mask.sum()} / {vegetation_mask.size} "
      f"({100 * vegetation_mask.mean():.1f}%)")

## Next steps

`reflectance`, `wavelengths`, and `vegetation_mask` from this notebook feed directly into `02_prospect_d_forward_model.ipynb`.